# RestoreAI — Dedicated Google Colab GPU Runtime for HYPIR
### Harnessing Diffusion-Yielded Score Priors for Image Restoration (SIGGRAPH 2025)

---

## 📌 Architecture & System Role
This notebook serves as the **dedicated GPU inference engine** for your local **RestoreAI** application:
1. **Cloud-Hosted Model Assets**: Pretrained HYPIR checkpoints (~800MB+ LoRA) and Stable Diffusion 2.1 base weights are downloaded and run **strictly inside this Colab runtime**, keeping your local computer clean.
2. **Official HYPIR Implementation**: Clones `https://github.com/XPixelGroup/HYPIR.git` and runs true generative diffusion score prior restoration on Colab's NVIDIA GPU.
3. **FastAPI Restoration Service**: Launches a RESTful server inside Colab exposing `/health`, `/api/restore`, `/api/restore/info`, and `/api/download/{filename}`.
4. **Secure Public HTTPS Tunnel**: Uses Cloudflare Tunnel (`trycloudflare.com`, zero token/signup needed) or `ngrok` to provide a secure public HTTPS endpoint.
5. **Local Web App Integration**: Your existing RestoreAI website (`http://localhost:5000`) sends user-uploaded images to this Colab API, receives high-fidelity restored images, and displays them on the interactive comparison slider.

> **⚠️ Critical Notice on Ephemeral Storage:**
> Google Colab runtime storage is **temporary**. When the runtime disconnects or resets, files in `/content` are deleted.
> Model checkpoints must be downloaded again when starting a new Colab session. Step 9 includes an optional Google Drive caching shortcut to bypass downloading in future sessions!

---

### ⚡ Quick Start: 3 Simple Steps
1. **Set GPU**: In Colab top menu, go to `Runtime` → `Change runtime type` → select **T4 GPU** → click **Save**.
2. **Run All Cells**: Press `Ctrl+F9` (or `Runtime` → `Run all`).
3. **Copy the Public URL**: In Step 6, copy the generated HTTPS URL (e.g. `https://xxx.trycloudflare.com`) and paste it into your local RestoreAI app at `http://localhost:5000/restore` (or your local `.env` file).

## Step 1: Verify Hardware & GPU Acceleration

In [ ]:
# Step 1: Verify GPU Acceleration in Colab
!nvidia-smi

import sys
import torch

print("=" * 60)
print(f"Python Version: {sys.version.split()[0]}")
print(f"PyTorch Version: {torch.__version__}")
cuda_ok = torch.cuda.is_available()
print(f"CUDA Available: {cuda_ok}")

if cuda_ok:
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Active GPU: {gpu_name} ({vram_gb:.2f} GB VRAM)")
    print("SUCCESS: GPU runtime confirmed ready for HYPIR inference.")
else:
    print("\n" + "!" * 60)
    print("WARNING: No GPU detected! HYPIR requires GPU acceleration.")
    print("Go to 'Runtime' -> 'Change runtime type' and select 'T4 GPU'.")
    print("!" * 60)
print("=" * 60)

## Step 2: Install HYPIR Compatible Dependencies
Installs Diffusers, Transformers, Accelerate, PEFT, FastAPI, Uvicorn, and Cloudflare Tunneling inside Colab only.

In [ ]:
# Step 2: Install official HYPIR and server dependencies
print("Installing dependencies... (this takes ~45 seconds)")

# 1. Upgrade pip to modern resolver with pre-compiled wheel support
!pip install --quiet --upgrade pip

# 2. Remove outdated torchao to eliminate diffusers quantization import bug
!pip uninstall -y torchao

# 3. Force pre-built binary wheels to avoid building tokenizers from Rust source
!pip install --quiet --prefer-binary tokenizers

# 4. Install core HYPIR and server packages
!pip install --quiet --prefer-binary \
    diffusers \
    transformers \
    accelerate \
    peft \
    omegaconf \
    einops \
    safetensors \
    scikit-image \
    fastapi \
    "uvicorn[standard]" \
    python-multipart \
    pycloudflared \
    nest-asyncio

# 5. Verify imports
import diffusers, transformers, accelerate, fastapi, pycloudflared
print(f"✓ Dependencies successfully installed!")
print(f"  - Diffusers: {diffusers.__version__}")
print(f"  - Transformers: {transformers.__version__}")
print(f"  - Accelerate: {accelerate.__version__}")
print(f"  - FastAPI: {fastapi.__version__}")

## Step 3: Clone Official HYPIR Repository (XPixelGroup)
Clones the official research repository directly inside `/content/HYPIR` in Colab.

In [ ]:
# Step 3: Clone official HYPIR repository
import os
import sys

HYPIR_DIR = "/content/HYPIR"
if not os.path.exists(HYPIR_DIR):
    print("Cloning XPixelGroup/HYPIR from GitHub...")
    !git clone https://github.com/XPixelGroup/HYPIR.git {HYPIR_DIR}
else:
    print(f"HYPIR repository already cloned at {HYPIR_DIR}")

# Add repository to Python sys.path so modules can be imported
if HYPIR_DIR not in sys.path:
    sys.path.insert(0, HYPIR_DIR)

# Verify repository files
assert os.path.exists(os.path.join(HYPIR_DIR, "test.py")), "HYPIR test.py not found in repo!"
print("HYPIR repository verified successfully.")

## Step 4: Download Pretrained HYPIR Weights & Base Model
Downloads the official `HYPIR_sd2.pth` LoRA checkpoint from Hugging Face (`lxq007/HYPIR`) into `/content/weights`.

In [ ]:
# Step 4: Download HYPIR Checkpoint from Hugging Face
import os
import urllib.request

WEIGHTS_DIR = "/content/weights"
os.makedirs(WEIGHTS_DIR, exist_ok=True)
WEIGHT_PATH = os.path.join(WEIGHTS_DIR, "HYPIR_sd2.pth")

HF_WEIGHT_URL = "https://huggingface.co/lxq007/HYPIR/resolve/main/HYPIR_sd2.pth"

if not os.path.exists(WEIGHT_PATH) or os.path.getsize(WEIGHT_PATH) < 100 * 1024 * 1024:
    print("Downloading HYPIR_sd2.pth (~850 MB) from Hugging Face...")
    !wget --progress=bar:force -O {WEIGHT_PATH} {HF_WEIGHT_URL}
else:
    print(f"HYPIR weights already present ({os.path.getsize(WEIGHT_PATH) / (1024**2):.1f} MB)")

weight_size_mb = os.path.getsize(WEIGHT_PATH) / (1024**2)
print(f"Weight checkpoint verified: {WEIGHT_PATH} ({weight_size_mb:.2f} MB)")
assert weight_size_mb > 50, "Weight file appears incomplete!"

## Step 5: Implement HYPIR Restoration API Server (FastAPI)
Builds a high-performance RESTful API service supporting:
- `GET /`: Health & service status
- `GET /health`: GPU status, VRAM availability, engine verification
- `GET /api/restore/info`: Supported parameters and upscale factors (2x, 3x, 4x)
- `POST /api/restore`: Multipart upload, HYPIR diffusion restoration, returns metadata & download link
- `GET /api/download/{filename}`: Restored image retrieval

In [ ]:
# Step 5: Create standalone Colab HYPIR FastAPI server
import os
import sys
import time
import shutil
import asyncio
import subprocess
from pathlib import Path
from typing import Optional

from fastapi import FastAPI, File, Form, UploadFile, HTTPException, status, Request
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse, JSONResponse
from PIL import Image
import torch

UPLOAD_DIR = Path("/content/server_uploads")
OUTPUT_DIR = Path("/content/server_outputs")
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if "/content/HYPIR" not in sys.path:
    sys.path.insert(0, "/content/HYPIR")

from torchvision import transforms
from HYPIR.enhancer.sd2 import SD2Enhancer

print("Loading official HYPIR SD2 model weights into GPU VRAM...")
hypir_model = SD2Enhancer(
    base_model_path="sd2-community/stable-diffusion-2-1-base",
    weight_path=WEIGHT_PATH,
    lora_modules=[
        "to_k", "to_q", "to_v", "to_out.0", "conv", "conv1", "conv2",
        "conv_shortcut", "conv_out", "proj_in", "proj_out", "ff.net.2", "ff.net.0.proj"
    ],
    lora_rank=256,
    model_t=200,
    coeff_t=200,
    device="cuda" if torch.cuda.is_available() else "cpu",
)
hypir_model.init_models()
print("✓ HYPIR model loaded and ready in GPU VRAM!")

app = FastAPI(
    title="RestoreAI HYPIR Colab GPU API",
    version="1.0.0",
    description="Dedicated GPU inference server for RestoreAI running official HYPIR diffusion score priors.",
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

gpu_lock = asyncio.Lock()

@app.get("/")
def index():
    return {
        "service": "RestoreAI HYPIR Colab GPU API",
        "status": "online",
        "health_check": "/health",
        "documentation": "/docs",
    }

@app.get("/health")
def health():
    cuda_available = torch.cuda.is_available()
    gpu_name = torch.cuda.get_device_name(0) if cuda_available else None
    vram_mb = round(torch.cuda.get_device_properties(0).total_memory / (1024*1024)) if cuda_available else 0
    return {
        "status": "ready",
        "active_engine": "HYPIR (Diffusion-Yielded Score Priors)",
        "hardware": {
            "cuda_available": cuda_available,
            "device": "cuda" if cuda_available else "cpu",
            "gpu_name": gpu_name,
            "gpu_vram_mb": vram_mb,
        },
        "hypir_weights_found": os.path.exists(WEIGHT_PATH),
    }

@app.get("/api/restore/info")
def model_info():
    return {
        "service": "RestoreAI HYPIR Colab GPU API",
        "model": "HYPIR (XPixelGroup SIGGRAPH 2025)",
        "supported_upscale_factors": [2, 3, 4],
        "default_patch_size": 512,
        "device": "cuda" if torch.cuda.is_available() else "cpu",
    }

@app.post("/api/restore")
async def restore_image(
    request: Request,
    image: UploadFile = File(...),
    upscale: int = Form(4),
    patch_size: int = Form(512),
    stride: int = Form(256),
    seed: int = Form(231),
    prompt: Optional[str] = Form(None),
):
    final_filename = f"restored_{token}.png"
    final_path = OUTPUT_DIR / final_filename

    # Load uploaded image directly into PIL
    input_pil = Image.open(image.file).convert("RGB")
    orig_w, orig_h = input_pil.size

    start_time = time.time()

    to_tensor = transforms.ToTensor()
    img_tensor = to_tensor(input_pil).unsqueeze(0)
    prompt_text = prompt.strip() if (prompt and prompt.strip()) else "a clean, high-resolution, sharp, detailed photograph, master quality"

    # Execute direct HYPIR GPU inference in-memory
    async with gpu_lock:
        with torch.no_grad():
            output_pil = hypir_model.enhance(
                lq=img_tensor,
                prompt=prompt_text,
                upscale=upscale,
                patch_size=patch_size,
                stride=stride,
                return_type="pil",
            )[0]

    output_pil.save(final_path, format="PNG")
    final_w, final_h = output_pil.size

    elapsed = round(time.time() - start_time, 2)
    base_url = str(request.base_url).rstrip("/")

    return {
        "success": True,
        "filename": final_filename,
        "download_url": f"{base_url}/api/download/{final_filename}",
        "original_dimensions": [orig_w, orig_h],
        "restored_dimensions": [final_w, final_h],
        "upscale_factor": upscale,
        "processing_time_sec": elapsed,
        "engine": "HYPIR (Diffusion-Yielded Score Prior)",
        "device": "cuda" if torch.cuda.is_available() else "cpu",
    }

@app.get("/api/download/{filename}")
def download_file(filename: str):
    safe_name = os.path.basename(filename)
    file_path = OUTPUT_DIR / safe_name
    if not file_path.exists():
        raise HTTPException(status_code=404, detail="File not found.")
    return FileResponse(str(file_path), media_type="image/png", filename=safe_name)

print("FastAPI restoration service defined successfully.")

## Step 6: Start Server & Expose via Public HTTPS Tunnel
Runs the FastAPI server locally on port 8000 and exposes it to the Internet using **Cloudflare Tunnel** (`trycloudflare.com`).
*(Zero signup or token required. A public HTTPS URL will be printed below.)*

In [ ]:
# Step 6: Launch Uvicorn Server in Background & Open Public HTTPS Tunnel
import time
import threading
import uvicorn
import requests
from pycloudflared import try_cloudflare

# 1. Clean up any existing process on port 8000 to ensure fresh code is loaded
!fuser -k 8000/tcp
time.sleep(1)

# 2. Start Uvicorn in a daemon thread with the updated FastAPI app
def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
print("Waiting for local server to boot...")
time.sleep(3)

# 2. Test local endpoint
try:
    local_res = requests.get("http://127.0.0.1:8000/health", timeout=3)
    print("Local server status:", local_res.json())
except Exception as e:
    print("Local test error:", e)

# 3. Open Cloudflare public tunnel
print("Opening Cloudflare HTTPS Tunnel (this may take 5-10 seconds)...")
public_url = try_cloudflare(port=8000).tunnel

print("\n" + "=" * 75)
print("🚀 HYPIR PUBLIC RESTORATION API IS ONLINE!")
print(f"Public HTTPS Endpoint: {public_url}")
print("=" * 75)
print("\n👉 HOW TO CONNECT TO YOUR EXISTING RESTOREAI WEBSITE:")
print(f"1. Open http://localhost:5000/restore in your browser")
print(f"2. Click '⚙️ Colab Tunnel URL', paste '{public_url}', and click 'Test & Connect'")
print("   OR update your local .env file:")
print(f"   RESTORE_AI_API_URL={public_url}")
print("=" * 75)

## Step 7: Verify Public Tunnel Health Endpoint

In [ ]:
# Step 7: Automated Public Health Check
import time
import requests

health_endpoint = f"{public_url}/health"
print(f"Pinging {health_endpoint}...")

start = time.time()
res = requests.get(health_endpoint, timeout=10)
latency = round((time.time() - start) * 1000, 1)

print(f"HTTP Status: {res.status_code}")
print(f"Latency: {latency} ms")
print("Response Data:")
import json
print(json.dumps(res.json(), indent=2))
assert res.status_code == 200, "Health check failed!"

## Step 8: End-to-End Image Restoration Test in Colab
Sends a test image to the running API, performs real GPU inference with HYPIR, downloads the restored result, and displays a Before vs After visual comparison.

In [ ]:
# Step 8: End-to-End Restoration Test
import io
import requests
import matplotlib.pyplot as plt
from PIL import Image, ImageFilter

# Create a degraded synthetic test image with noise and blur
test_img = Image.new("RGB", (256, 256), color=(140, 100, 180))
for x in range(30, 226, 20):
    for y in range(30, 226, 20):
        test_img.putpixel((x, y), (255, 220, 100))
test_img = test_img.filter(ImageFilter.GaussianBlur(radius=1.5))

buf = io.BytesIO()
test_img.save(buf, format="PNG")
buf.seek(0)

print("Submitting test image to Colab HYPIR API...")
files = {"image": ("test_input.png", buf, "image/png")}
data = {"upscale": "2", "patch_size": "512", "stride": "256", "seed": "231"}

restore_res = requests.post(f"{public_url}/api/restore", files=files, data=data, timeout=120)
print(f"Restore Response Status: {restore_res.status_code}")
res_data = restore_res.json()
print("Restoration Output Metadata:", res_data)

assert res_data.get("success"), f"Restoration failed: {res_data}"

# Download restored result image
download_url = res_data["download_url"]
img_res = requests.get(download_url)
restored_img = Image.open(io.BytesIO(img_res.content))

# Plot Before vs After
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(test_img)
axes[0].set_title(f"Input Degraded Image\n({test_img.size[0]}x{test_img.size[1]})")
axes[0].axis("off")

axes[1].imshow(restored_img)
axes[1].set_title(f"HYPIR Restored Output\n({restored_img.size[0]}x{restored_img.size[1]}) - {res_data['processing_time_sec']}s")
axes[1].axis("off")
plt.tight_layout()
plt.show()

print("\nSUCCESS: End-to-end HYPIR restoration verified successfully on Colab GPU!")

## Step 9: Reconnection, Ephemeral Storage & Production Tips

### 1. What happens if Colab disconnects or the tunnel URL changes?
- If Colab disconnects, re-run **Step 6** and **Step 7**.
- A fresh `https://xxx.trycloudflare.com` URL will be generated.
- On your RestoreAI website (`http://localhost:5000/restore`), click **⚙️ Colab Tunnel URL**, paste the new URL, and click **Test & Connect**. The website immediately begins using the new tunnel without restarting Flask!

### 2. How to avoid re-downloading model weights upon runtime reset
If you have Google Drive space, you can save `HYPIR_sd2.pth` to your Drive once and load it instantly in seconds:
```python
from google.colab import drive
drive.mount('/content/drive')
# Backup to Drive:
# !cp /content/weights/HYPIR_sd2.pth /content/drive/MyDrive/HYPIR_sd2.pth
# In future sessions, copy from Drive in 10 seconds:
# !cp /content/drive/MyDrive/HYPIR_sd2.pth /content/weights/HYPIR_sd2.pth
```

### 3. GPU VRAM Memory Management
If you encounter a CUDA Out-of-Memory (OOM) error with large images, lower the patch size in the API call from `512` to `256` or run:
```python
import torch
torch.cuda.empty_cache()
```